In [1]:
import os
import sys

# Replace these paths with the actual path to Java on your machine
os.environ["JAVA_HOME"] = "/home/jbisson/.sdkman/candidates/java/17.0.15-librca/"
os.environ["SPARK_LOCAL_IP"]= "127.0.0.1"

# Python executable path (helps Spark find uv's virtual env python)
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable


# Replace these with your actual Apache Polaris details
CATALOG_NAME = "test_catalog_mixed"                     # The catalog name defined in Polaris
NAMESPACE_NAME = "test_namespace2"                      # The namespace you want to use in Polaris
CLIENT_ID = "root"                                      # The Principal Client ID
CLIENT_SECRET = "s3cr3t"                                # The Principal Client Secret
REALM = "realm-mixed"                                   # Apache Polaris REALM

In [2]:
%%bash -s "$CATALOG_NAME" "$NAMESPACE_NAME" "$REALM" "$CLIENT_ID" "$CLIENT_SECRET"

CATALOG_NAME=$1
NAMESPACE_NAME=$2
REALM=$3
CLIENT_ID=$4
CLIENT_SECRET=$5

./../../../polaris catalogs create \
  --storage-type file \
  --default-base-location file:///tmp/${CATALOG_NAME} \
  ${CATALOG_NAME} \
  --host 127.0.0.1 \
  --port 8181 \
  --client-id ${CLIENT_ID} \
  --client-secret ${CLIENT_SECRET} \
  --realm ${REALM}



In [3]:
%%bash -s "$CATALOG_NAME" "$NAMESPACE_NAME" "$REALM" "$CLIENT_ID" "$CLIENT_SECRET"

CATALOG_NAME=$1
NAMESPACE_NAME=$2
REALM=$3
CLIENT_ID=$4
CLIENT_SECRET=$5

./../../../polaris namespaces create \
  --catalog $CATALOG_NAME \
  --location file:///tmp/$CATALOG_NAME/$NAMESPACE_NAME \
  $NAMESPACE_NAME \
  --host 127.0.0.1 \
  --port 8181 \
  --client-id ${CLIENT_ID} \
  --client-secret ${CLIENT_SECRET} \
  --realm ${REALM}

In [4]:
from pyspark.sql import SparkSession

POLARIS_URI = f"http://localhost:8181/api/catalog/"     # Your Polaris REST endpoint

# Create a local Spark session
spark = ( 
    SparkSession.builder
    .appName("LocalPySparkSession")
    .master("local[*]")
    # 1 Download the iceberg spark runtime dependencies
    .config("spark.jars.packages", "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.11.0")
    # 2. Enable Iceberg Extensions
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    # 3. Define the 'test_catalog' as a Polaris catalog running locally
    .config(f"spark.sql.catalog.{CATALOG_NAME}", "org.apache.iceberg.spark.SparkCatalog")
    .config(f"spark.sql.catalog.{CATALOG_NAME}.catalog-impl", "org.apache.iceberg.rest.RESTCatalog") \
    .config(f"spark.sql.catalog.{CATALOG_NAME}.uri", POLARIS_URI)
    .config(f"spark.sql.catalog.{CATALOG_NAME}.warehouse", CATALOG_NAME)
    .config(f"spark.sql.catalog.{CATALOG_NAME}.header.X-Iceberg-Access-Realm", REALM)
    .config(f"spark.sql.catalog.{CATALOG_NAME}.header.Polaris-Realm", REALM) \

    # 4. Configure Authentication (Username and Password)
    # The standard Iceberg REST convention packs them into a 'username:password' credential string
    .config(f"spark.sql.catalog.{CATALOG_NAME}.scope", "PRINCIPAL_ROLE:ALL")
    .config(f"spark.sql.catalog.{CATALOG_NAME}.credential", f"{CLIENT_ID}:{CLIENT_SECRET}")
    .config(f"spark.sql.catalog.{CATALOG_NAME}.auth.header.X-Iceberg-Access-Realm", REALM)
    .config(f"spark.sql.catalog.{CATALOG_NAME}.auth.header.Polaris-Realm", REALM)
    .config(f"spark.sql.catalog.{CATALOG_NAME}.oauth2-server-uri", "http://localhost:8181/api/catalog/v1/oauth/tokens")
    .config(f"spark.sql.catalog.{CATALOG_NAME}.rest.auth.type", "oauth2")
    # 5. Set test_catalog as the default catalog for this session
    .config("spark.sql.defaultCatalog", CATALOG_NAME)
    # 6. Configure THE POLARIS REALM
    
    .getOrCreate()
)
# Verify the session is working
print(f"Spark Version: {spark.version}")
print(f"Spark App Name: {spark.sparkContext.appName}")

# Quick sanity check: Create a tiny DataFrame
data = [("Alice", 25), ("Bob", 30), ("Charlie", 35)]
columns = ["Name", "Age"]
df = spark.createDataFrame(data, schema=columns)
df.show()

:: loading settings :: url = jar:file:/home/jbisson/github/polaris/runtime/polaris-self-serve/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/jbisson/.ivy2/cache
The jars for the packages stored in: /home/jbisson/.ivy2/jars
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-f3e40ec2-6b84-4ada-90fc-689d03db6a91;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.11.0 in central
:: resolution report :: resolve 105ms :: artifacts dl 3ms
	:: modules in use:
	org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.11.0 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   1   |   0   |   0   |   0   ||   1   |   0   |
	---------------------------------------------------------------------
:: retrieving :: org.apach

Spark Version: 3.5.6
Spark App Name: LocalPySparkSession


+-------+---+
|   Name|Age|
+-------+---+
|  Alice| 25|
|    Bob| 30|
|Charlie| 35|
+-------+---+



In [5]:
spark.sql(f"SHOW NAMESPACES IN {CATALOG_NAME}").show(truncate=False)
spark.sql(f"DESCRIBE NAMESPACE EXTENDED {CATALOG_NAME}.{NAMESPACE_NAME}").show(truncate=False)

+---------------+
|namespace      |
+---------------+
|test_namespace2|
+---------------+

+--------------+----------------------------------------------------------------------------------------------------------+
|info_name     |info_value                                                                                                |
+--------------+----------------------------------------------------------------------------------------------------------+
|Catalog Name  |test_catalog_mixed                                                                                        |
|Namespace Name|test_namespace2                                                                                           |
|Location      |file:///tmp/test_catalog_mixed/test_namespace2/                                                           |
|Properties    |((polaris_owner,root), (polaris_owner_created_at,2026-06-05T17:34:48.746551629Z), (polaris_owner_id,root))|
+--------------+-------------------------

In [6]:
# Quick sanity check: Create a tiny DataFrame
data = [("Alice", 25), ("Bob", 30), ("Charlie", 35)]
columns = ["Name", "Age"]
df = spark.createDataFrame(data, schema=columns)
# Define the full table path
table_path = f"{CATALOG_NAME}.{NAMESPACE_NAME}.test_table"

# Write and create the table
df.writeTo(table_path).create()

26/06/05 13:35:00 WARN RESTMetricsReporter: Failed to report metrics to REST endpoint v1/test_catalog_mixed/namespaces/test_namespace2/tables/test_table/metrics
org.apache.iceberg.exceptions.RESTException: Unable to process (code: 404, type: NoSuchTableException): Table does not exist: test_namespace2.test_table
	at org.apache.iceberg.rest.ErrorHandlers.createRESTException(ErrorHandlers.java:115)
	at org.apache.iceberg.rest.ErrorHandlers$DefaultErrorHandler.accept(ErrorHandlers.java:357)
	at org.apache.iceberg.rest.ErrorHandlers$DefaultErrorHandler.accept(ErrorHandlers.java:321)
	at org.apache.iceberg.rest.HTTPClient.throwFailure(HTTPClient.java:242)
	at org.apache.iceberg.rest.HTTPClient.execute(HTTPClient.java:347)
	at org.apache.iceberg.rest.HTTPClient.execute(HTTPClient.java:299)
	at org.apache.iceberg.rest.BaseHTTPClient.post(BaseHTTPClient.java:112)
	at org.apache.iceberg.rest.RESTClient.post(RESTClient.java:150)
	at org.apache.iceberg.rest.RESTMetricsReporter.lambda$report$1(RES

In [7]:
# Quick sanity check: Create a tiny DataFrame
data = [("Alice", 15), ("Bob", 20), ("Charlie", 55)]
columns = ["Name", "Age"]
df = spark.createDataFrame(data, schema=columns)
# Define the full table path
table_path = f"{CATALOG_NAME}.{NAMESPACE_NAME}.test_table"

# Write and create the table
df.writeTo(table_path).append()

In [8]:
spark.sql(f"SELECT * FROM {CATALOG_NAME}.{NAMESPACE_NAME}.test_table").show(truncate=False)

+-------+---+
|Name   |Age|
+-------+---+
|Alice  |15 |
|Alice  |25 |
|Bob    |30 |
|Bob    |20 |
|Charlie|35 |
|Charlie|55 |
+-------+---+



In [9]:
spark.sql(f"DESC EXTENDED {CATALOG_NAME}.{NAMESPACE_NAME}.test_table").show(truncate=False)

+-----------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------+
|col_name                     |data_type                                                                                                                                                                                                                                                        |comment|
+-----------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------+
|Name                         |string                                                                     

In [10]:
spark.sql(f"""
ALTER NAMESPACE {CATALOG_NAME}.{NAMESPACE_NAME} SET DBPROPERTIES ('key1' = 'value1', 'key2' = 'value2')
""").show(truncate=False)

++
||
++
++



In [11]:
spark.sql(f"""
ALTER TABLE {CATALOG_NAME}.{NAMESPACE_NAME}.test_table 
SET TBLPROPERTIES (
    'prop1' = 'parquet3',
    'prop2' = 'zstd2',
    'prop3' = 'zstd1'
)
""").show(truncate=False)

++
||
++
++



In [12]:
spark.sql(f"""
ALTER TABLE {CATALOG_NAME}.{NAMESPACE_NAME}.test_table 
SET TBLPROPERTIES (
    'prop1_a' = 'parquet3',
    'prop2_b' = 'zstd2',
    'prop3_c' = 'zstd1'
)
""").show(truncate=False)

++
||
++
++

